# GARTEUR tap tests: EMA and OMA with modekit

Tap tests on the aluminium GARTEUR structure, healthy, damaged (loosened bolts) and restored. The first part is an experimental modal analysis of the hammer tests: input and response checks, force and exponential windows, FRFs and coherence, then band-wise pLSCF and LSFD. The second part treats the fan-excited records as output-only data (OMA).

The measurements come from the [modekit-data](https://github.com/garavanis/modekit-data) repository; the notebook downloads the sets it needs on its first run.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import jax.numpy as jnp

from modekit.dataprep import taptest

Let's load the data

In [ ]:
from pathlib import Path
from modekit import datasets

# the test sets this notebook uses, downloaded on the first run from the
# modekit-data repository into data/ next to it
TAP_SETS = [
    "fr_tap_hs1", "fr_tap_hs2", "fr_tap_hsr", "fr_tap_ds",
    "rw_tap_hs1", "rw_tap_hs2", "rw_tap_hsr", "rw_tap_ds",
    "fanfr_nb_hs", "fanfr_nb_ds", "fanfr_b0_hs", "fanfr_b0_ds", "fanfr_b45_hs", "fanfr_b45_ds",
]
HERE = Path.cwd()   # the examples folder: Jupyter runs a notebook in its own folder
DATA_DIR = datasets.fetch("heartspace", [f"{s}.csv" for s in TAP_SETS], HERE / "data")
OUT_DIR = HERE / "outputs" / "garteur"   # tables and the MPE cache
FIG_DIR = HERE / "figures" / "garteur"
data_loader = taptest.DataLoader(DATA_DIR)
data = data_loader.load_data()

# EMA

In [ ]:
# save in dictionary for use throughout
taps = {
    "fr_hs1": data["fr_tap_hs1"],
    "fr_hs2": data["fr_tap_hs2"],
    "fr_hsr": data["fr_tap_hsr"],
    "rw_hs1": data["rw_tap_hs1"],
    "rw_hs2": data["rw_tap_hs2"],
    "rw_hsr": data["rw_tap_hsr"],
    "fr_ds":  data["fr_tap_ds"],
    "rw_ds":  data["rw_tap_ds"],
}

# the response channels, in data order (Front_255, Tail_264, right_wing_484, left_wing_258)
SENSORS = ["nose", "tail", "starboard", "port-side"]

In [ ]:
taps["fr_hs1"].load_channel("Hammer").shape

In [ ]:
taps["fr_hs1"].data.shape

We observe that multiple impacts occur in the 4th repetition of the *fr_tap_hs1* set. 
We choose to discard it for our analysis to retain clean input spectra. 
More details in the *"Input power spectra"* section

In [ ]:
from modekit.plots import plot_signal

plt.style.use("modekit.mathtext")  # Computer Modern maths, no LaTeX needed; modekit.latex with a LaTeX installation

fs = taps["fr_hs1"].sample_rate
print(f"Sample rate: {fs} Hz")

# Time vector from the sample rate (shared across all tap sets since we trimmed to the shortest rep)
nt = taps["fr_hs1"].data.shape[2]
time_array = np.arange(nt) / fs

hammer_hs1 = taps["fr_hs1"].load_channel("Hammer")   # (reps, nt)
rep_labels = [rf"$f_{{rep {r}}}(t)$ [N]" for r in range(1, hammer_hs1.shape[0] + 1)]
fig, axes = plot_signal(
    time_array,
    hammer_hs1,
    x_label="Time [s]",
    y_labels=rep_labels,
    figsize=(6, 10),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1.pdf",
)

Since the impulse is a sub-millisecond spike we can zoom in to see the pulse shape

In [ ]:
fig, axes = plot_signal(time_array, hammer_hs1, x_label="Time [s]",
    y_labels=rep_labels, xlim=(0.1, 0.5), figsize=(4, 8),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1_pulse.pdf")

We can zoom in further to see the level of noise in the signal

In [ ]:
_, axes = plot_signal(time_array, hammer_hs1, x_label="Time [s]",
    y_labels=rep_labels, xlim=(1.5, 1.525), ylim=(-0.15, 0.15), figsize=(4, 8),
    save_fig_name=FIG_DIR / "ema/hammer_fr_hs1_noise.pdf")

Similarly, we can observe the other sets. We do not observe any multiple impact issues (uncomment to see the plots)

In [ ]:
# # Uncomment  to plot the tap test sets

# figs = {}
# for name, tap in taps.items():
#     fig, axes = plot_signal(
#         time_array,
#         tap.load_channel("Hammer"),
#         x_label="Time [s]",
#         y_labels=[rf"$f_{{rep {r}}}(t)$ [N]" for r in range(1, len(tap.load_channel("Hammer")) + 1)],
#         figsize=(6, 10),
#     )
#     fig.suptitle(name)
#     figs[name] = (fig, axes)

# # Per-plot axis tweaks 
# figs["rw_hs1"][1][0].set_ylim(-1, 1); figs["rw_hs1"][1][0].set_xlim(0.9, 1)
# figs["rw_hs2"][1][1].set_ylim(-2, 1)

Apart from visual inspection we can make use of the `taptest.find_bad_reps` to identify double impacts or clipping of the measured signals.

In [ ]:
# flag bad reps on each tap set
for name, tap in taps.items():
    bad_f = taptest.find_bad_reps(x=tap.load_channel("Hammer"), fs=fs, signal='i', print_summary=True, name=name)
    bad_r = taptest.find_bad_reps(x=tap.data[:, 1:, :],         fs=fs, signal='o', print_summary=True, name=name)
    tap.bad_reps = bad_f.any(axis=1) | bad_r.any(axis=1)

if not any(tap.bad_reps.any() for tap in taps.values()):
    print("All sets clean")

#### Input power spectra
We compute the power spectrum of the input excitation via the FFT for each rep of each tap-test set. \
We then average these spectra across the repetitions in a set to reduce the random error.

We compute the spectrum and not the density (PSD) for the force signal. PSD assumes a signal is continuous and stationary. On the other hand, a hammer impact is a transient event. Calculating a PSD would incorrectly average the impact's finite energy over the entire recording window. With the power spectrum instead, we capture the true energy delivered to the system regardless of the record length.

### Front

In [ ]:
from modekit.freq_models import EmaModel
import equinox as eqx

tap_models = {
    name: EmaModel(
        X=taps[name].load_channel("Hammer")[:, None, :],
        Y=taps[name].data[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name in taps
}

print("\nEmaModel fields:")
print("-" * 30)
eqx.tree_pprint(tap_models["fr_hs1"])

In [ ]:
from collections import namedtuple

# called psd for convenience but here it is actually power spectra
PSD = namedtuple("PSD", "freqs Sxx Syy")
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

We plot the averaged input power spectra for the three *front* tap-test sets.

As expected, the spectra roll off with frequency: the soft hammer tip concentrates the impact energy at low frequencies, so above roughly **2000 Hz** the force spectrum is essentially noise. 

In the zoomed-in view, the *fr_tap_hs1* set (green) shows a characteristic **comb-filter** pattern: the regularly spaced notches caused by the double impact in the 4th tap of this set.

In [ ]:
fig,axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front.pdf",
)

# zoomed-view
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    xlim=(0, 800),
    ylim=(1e-6, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_zoom.pdf",
)
plt.show()

Let's see what happens to the green line when we discard the 4th rep from the *fr_tap_hs1* set. 

In [ ]:
# rebuild tap_models["fr_hs1"]
tap_models["fr_hs1"] = EmaModel(X=taps["fr_hs1"].load_channel("Hammer", good=True)[:, None, :], 
                                Y=taps["fr_hs1"].good[:, 1:, :], fs=fs, exc_type="transient")
# rebuild psd["fr_hs1"] from the updated model
psd["fr_hs1"] = PSD(*tap_models["fr_hs1"].get_psd())

In [ ]:
# Plot again 
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    xlim=(0, 800),
    ylim=(1e-6, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_zoom_good.pdf",
)
plt.show()

### Right wing

In [ ]:
eqx.tree_pprint(tap_models["rw_hsr"])

In [ ]:
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Sxx,
        "R-wing tap HS2": psd["rw_hs2"].Sxx,
        "R-wing tap HSR": psd["rw_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    # xlim=(0, 800),
    # ylim=(1e-7, 2e-2),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_rwing.pdf",
)
plt.show()

### Applying force windowing

The use of a force window is to minimize the effect of spurious noise on the input channel. 

This window is typically implemented as a rectangular window that exists over a portion of the block or it may be a cosine wave taper on the input, depending on the particular implementation. [p. 137, Avitabile2018]

Here we choose the cosine wave taper. 

In [ ]:
from modekit.dataprep import windows

def _fw(x, window=None, enabled=True):
    if window is None:
        window = windows.Tukey(pre_ms=5.0, post_ms=5.0, taper_ms=2.0)
        
    return window.apply(x, fs=fs) if enabled else x

# rebuild every model with the (windowed) input, dropping bad reps per set
tap_models = {
    name: EmaModel(
        X=_fw(tap.load_channel("Hammer", good=True))[:, None, :],
        Y=tap.good[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name, tap in taps.items()
}
# rebuild the power spectra from the updated models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# FRONT
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Sxx,
        "Front tap HS2": psd["fr_hs2"].Sxx,
        "Front tap HSR": psd["fr_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_front_force_win.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Sxx,
        "R-wing tap HS2": psd["rw_hs2"].Sxx,
        "R-wing tap HSR": psd["rw_hsr"].Sxx,
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{xx}(f)$ [N$^2$]"],
    yscale="log",
    figsize=(6, 4),
    save_fig_name=FIG_DIR / "ema/Sxx_rwing_force_win.pdf",
)
plt.show()

Let's plot the response signals

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in taps.items():
for name, tap in {k: taps[k] for k in ["fr_hs1"]}.items():
    n_reps = tap.data.shape[0]
    n_cols = len(tap.channels[1:])

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(tap.channels[1:]):
        sig = tap.load_channel(ch)
        color = cmap(j)   # one color per sensor 
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0)
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    fig.suptitle(name)
    fig.tight_layout()
    save_path = FIG_DIR / f"ema/responses_{name}.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

In [ ]:
# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

The response signal of left_wing_258 at rep 5 of *rw_tap_hs1* set is identified as bad (glitch).\
We want to discard it. 

In [ ]:
# find_bad_reps doesn't catch this kind of glitch,
# flag rep 5 of rw_hs1 manually before rebuilding
taps["rw_hs1"].flag(4)

# rebuild again dropping rep 5 of rw_tap_hs1
tap_models = {
    name: EmaModel(
        X=_fw(tap.load_channel("Hammer", good=True))[:, None, :],
        Y=tap.good[:, 1:, :],
        fs=fs,
        exc_type="transient",
    )
    for name, tap in taps.items()
}
# rebuild the power spectra from the updated models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# Plot again

# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}_good.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}_good.pdf",
)
plt.show()

### Exponential Window on Responses

Lightly damped structures often continue to ring after the recording block ends, which causes spectral leakage. However, applying an exponential window to fix this is a double-edged sword.

**The Trade-off:**
* **Unwindowed:** We risk spectral leakage, but we preserve the true physical signal, giving you the best chance to observe all modes (especially closely spaced ones).
* **Windowed:** We eliminate leakage, but we introduce artificial damping that must be mathematically corrected after extracting the poles.

We shouldn't apply an exponential window blindly. Instead we can:
1. Check if response signal naturally decays to zero before the block ends.
2. Evaluate the FRF & Coherence (look for symptoms like smeared peaks or sharp coherence drops at resonance). 
3. Apply the exponential window if there is leakage.

But first we need to choose the decay $\alpha$.
The general recommendation is to apply as little decay as possible (p. 150: "Generally, try to add as little damping to the measurement as possible"). \
One practical approach is to size $\alpha$ for the worst case: identify the signal that has decayed the least and set $\alpha$ so that it reaches a desired level by the end of the window. \

Let's see effect of the damping window. There is also the option to completely zero out the pre-trigger.

Pre-trigger zeroing removes numerical noise from an otherwise clean measurement (enforcing the zero-initial-condition assumption of the transfer function $H(s) = X(s)/F(s)$, p. 281), but it does not extinguish actual ring-down. That contamination persists in the post-impact region and the only remedy is sufficient dwell time between taps.

In [ ]:
# Each tap set is an independent SIMO, so size its own exponential window.
# tap.good already drops the bad reps, so alpha is sized on clean reps only.
resp_win = {
    name: windows.Exponential.from_tap_tests(
        tap.good,
        fs,
        target_x=0.01,
        pre_ms=1.0,
        tail_frac=0.05,
        smooth_ms=20.0,
        zero_pretrigger=False,
    )
    for name, tap in taps.items()
}


def _rw(x, ref, name, windows=resp_win, enabled=True):
    """Apply the exponential window sized for tap set `name`."""
    return windows[name].apply(x, fs=fs, ref=ref) if enabled else x


for name, win in resp_win.items():
    print(f"{name}: response window sized (alpha={win.alpha:.4f} 1/s)")

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in taps.items():
for name, tap in {k: taps[k] for k in ["rw_ds"]}.items():
    n_reps = tap.data.shape[0]
    n_cols = len(tap.channels[1:])
    hammer = tap.load_channel("Hammer")  # reference for the exponential window

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(tap.channels[1:]):
        sig = tap.load_channel(ch)
        sig_win = _rw(sig, ref=hammer, name=name)  # exponentially decayed response
        color = cmap(j)   # one color per sensor
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0,
                            alpha=0.4, label="Original")
            axes[i, j].plot(time_array, sig_win[i, :], color=color, linewidth=1.0,
                            label="Windowed")
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    axes[0, 0].legend(loc="upper right", fontsize="small")
    fig.suptitle(name)
    fig.tight_layout()
    save_path = FIG_DIR / f"ema/responses_{name}_exp_win.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

In [ ]:
# build the new EmaModel here: force window on the input, exponential
# window on the responses
tap_models = {}
for name, tap in taps.items():
    hammer = tap.load_channel("Hammer", good=True)
    Y_raw = tap.good[:, 1:, :]
    # _rw works per (n_reps, n_samples) channel, so window each and stack back
    Y_win = np.stack(
        [_rw(Y_raw[:, c, :], ref=hammer, name=name, enabled=True) for c in range(Y_raw.shape[1])],
        axis=1, 
    )
    tap_models[name] = EmaModel(
        X=_fw(hammer)[:, None, :],
        Y=Y_win,
        fs=fs,
        exc_type="transient",
    )

# rebuild the power spectra from the windowed models
psd = {name: PSD(*m.get_psd()) for name, m in tap_models.items()}

In [ ]:
# Plot again

# 0 -> "Front_255"
# 1 -> "Tail_264"
# 2 -> "right_wing_484"
# 3 -> "left_wing_258"
sid = 2 

# FRONT TAP
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "Front tap HS1": psd["fr_hs1"].Syy[sid:sid + 1],
        "Front tap HS2": psd["fr_hs2"].Syy[sid:sid + 1],
        "Front tap HSR": psd["fr_hsr"].Syy[sid:sid + 1],
        "Front tap DS": psd["fr_ds"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 250),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_front_{SENSORS[sid]}_exp_win.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    psd["fr_hs1"].freqs,
    {
        "R-wing tap HS1": psd["rw_hs1"].Syy[sid:sid + 1],
        "R-wing tap HS2": psd["rw_hs2"].Syy[sid:sid + 1],
        "R-wing tap HSR": psd["rw_hsr"].Syy[sid:sid + 1],
        "R-wing tap DS": psd["rw_ds"].Syy[sid:sid + 1],
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$S_{yy}(f)$ [(m/s$^2$)$^2$]"],
    yscale="log",
    xlim=(0, 250),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/Syy_rwing_{SENSORS[sid]}_exp_win.pdf",
)
plt.show()

In [ ]:
# Each tap set is an independent SIMO test (P=1, Q=4), so we compute its FRF on its
# own model. 
FRF = namedtuple("FRF", "freqs H")
frf = {name: FRF(*m.get_frf(method="H1")) for name, m in tap_models.items()}

In [ ]:
# Plot FRF magnitude 
sid = 2
# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.abs(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.abs(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.abs(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.abs(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$|H|_{\mathrm{avg}}$ [(m/s$^2$)/N]"],
    yscale="log",
    xlim=(0, 800),
    # ylim=(5e-2, 1e1),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_mag_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.abs(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.abs(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.abs(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.abs(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$|H|_{\mathrm{avg}}$ [(m/s$^2$)/N]"],
    yscale="log",
    xlim=(0, 800),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_mag_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF real part: it crosses through zero at the natural frequency
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.real(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.real(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.real(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.real(frf["fr_ds"].H[sid])
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\Re(H)$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(-2.5, 1.5),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_real_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.real(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.real(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.real(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.real(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\Re(H)$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(-200,200), 
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_real_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF imaginary part: it reaches its peak at the natural frequency
sid = 3 

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        "Front tap HS1": np.imag(frf["fr_hs1"].H[sid]),
        "Front tap HS2": np.imag(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.imag(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.imag(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\Im(H)$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    ylim=(-3, 3),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_imag_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.imag(frf["rw_hs1"].H[sid]),
        "R-wing tap HS2": np.imag(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.imag(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.imag(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\Im(H)$ [(m/s$^2$)/N]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_imag_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

In [ ]:
# Plot the FRF phase
sid = 2

# FRONT TAP
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "Front tap HS1": np.angle(frf["fr_hs1"].H[sid]),
        # "Front tap HS2": np.angle(frf["fr_hs2"].H[sid]),
        "Front tap HSR": np.angle(frf["fr_hsr"].H[sid]),
        "Front tap DS": np.angle(frf["fr_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\angle H$ [rad]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_phase_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    frf["fr_hs1"].freqs,
    {
        # "R-wing tap HS1": np.angle(frf["rw_hs1"].H[sid]),
        # "R-wing tap HS2": np.angle(frf["rw_hs2"].H[sid]),
        "R-wing tap HSR": np.angle(frf["rw_hsr"].H[sid]),
        "R-wing tap DS": np.angle(frf["rw_ds"].H[sid]),
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\angle H$ [rad]"],
    yscale="linear",
    xlim=(0, 300),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/H_phase_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

### Coherence

In [ ]:
COH = namedtuple("COH", "freqs gamma")
coh = {name: COH(*m.get_coherence(method="ordinary")) for name, m in tap_models.items()}

In [ ]:
# Plot the coherence
sid = 3

# FRONT TAP
fig, axes = plot_signal(
    coh["fr_hs1"].freqs,
    {
        "Front tap HS1": coh["fr_hs1"].gamma[sid],
        "Front tap HS2": coh["fr_hs2"].gamma[sid],
        "Front tap HSR": coh["fr_hsr"].gamma[sid],
        "Front tap DS": coh["fr_ds"].gamma[sid]
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\gamma^2$ [-]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(0, 1.05),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/coherence_front_{SENSORS[sid]}.pdf",
)
plt.show()

# RIGHT WING
fig, axes = plot_signal(
    coh["fr_hs1"].freqs,
    {
        "R-wing tap HS1": coh["rw_hs1"].gamma[sid],
        "R-wing tap HS2": coh["rw_hs2"].gamma[sid],
        "R-wing tap HSR": coh["rw_hsr"].gamma[sid],
        "R-wing tap DS": coh["rw_ds"].gamma[sid]
    },
    x_label="Frequency [Hz]",
    y_labels=[r"$\gamma^2$ [-]"],
    yscale="linear",
    xlim=(0, 300),
    # ylim=(0, 1.05),
    figsize=(6, 4),
    save_fig_name=FIG_DIR / f"ema/coherence_rwing_{SENSORS[sid]}.pdf",
)
plt.show()

## Modal parameter estimation (MPE) via pLSCF

Generally, when performing modal parameter estimation (MPE), working in small frequency bands—each containing only a few modes—is desirable.

Poly-reference Least-Squares Complex Frequency (pLSCF / "PolyMAX") fits a discrete-time model. The basis is $z^k$, where $z = e^{j\omega\,dt}$ with $dt = 1/f_s$. At $f_s = 16384$ Hz, the time step $dt$ is so small that even the highest frequency in our band of interest (up to ~800 Hz) produces only a tiny change in $z$. The entire band of interest therefore occupies a microscopic arc of the unit circle near $z = 1$, and all basis vectors $z^0, z^1, \ldots, z^p$ crowd into the same neighbourhood. Because the fitting algorithm cannot distinguish them, it fails to find any poles within the band.

Cropping the frequency arrays to the band alone does not help: $dt$ is unchanged, so the mapping from frequency to $z$ is unchanged and the crowding persists.

The fix is to tell pLSCF to use a fictitious sampling rate $f_s^{\mathrm{fake}}$ set just above each sub-band's upper edge, for example, 300 Hz for a 2–120 Hz band. The fictitious time step $dt_{\mathrm{fake}} = 1/f_s^{\mathrm{fake}}$ is much larger, so the same frequency range now maps to a much wider arc of the unit circle and the basis vectors spread out, making the fit well-conditioned. Once poles are found on this stretched axis, they are converted back to physical frequencies and damping using the true $dt$ via $s = \ln(z)/dt$, so the identified natural frequencies and damping ratios are correct.

This is the subdomain curve-fitting approach of Avitabile (Ch. 5 and 9): fit small frequency bands, each containing only a few modes, then stitch the results together at the end.

In [ ]:
from modekit.bandmpe import full_band_mpe
from modekit import reporting as rp
from modekit.export import save_table

# --- MPE settings (part of the cache identity: change any -> refit) ---
# Soft criteria / Hard criteria
SC = {"err_fn": 0.01, "err_zeta": 0.05, "err_phi": 0.05}
HC = {"zeta_max": 0.1, "mpc_lim": 0.5, "mpd_lim": 0.5}

# (f_lo, f_hi, fs_fake): each band holds a few modes; fs_fake ~ 2.5*f_hi
BANDS = [(2, 120, 300.0), (120, 320, 800.0), (320, 560, 1400.0)]
ORDMAX = 100
EPS, MIN_PTS = 0.02, 8   # DBSCAN: modal-distance radius and core-point threshold

MPE_CACHE_DIR = OUT_DIR / "mpe_cache"
REFIT_MPE = False   # set True to ignore existing MPE cache files

`full_band_mpe` (in `modekit.bandmpe`) runs the whole procedure on one tap set:

1. **Band-wise pLSCF.** One pLSCF per sub-band of `BANDS`, at that band's fake sampling rate. The stable poles of a band are clustered into modes (DBSCAN with `EPS` and `MIN_PTS`, see below), and the modes of all the bands are combined into one pole set.
2. **LSFD.** One residue fit over the full band at the combined poles (see the LSFD section below). The hammer is the only reference, so the band fits already give each pole's participation factor. The LSFD keeps these factors fixed and solves only for the mode shapes (`refs="global"`). This is the second step of PolyMAX.

The band fits are cached as equinox files under `outputs/garteur/mpe_cache` (through `modekit.serialisation`), so reruns load them from disk. The name of a cache file carries a hash of the data and of the settings above, so changing either one refits. Set `REFIT_MPE = True` to refit anyway.

### MPE for Front taps

Every front set is fitted in the next cell, and all of its figures are saved under `figures/garteur/ema/<set>/`. Only the figures of the set named by `tap_id` are shown; change `tap_id` to view another set.

In [ ]:
# the procedure above on every front set: one pass, the hammer is the only reference
FR_TAPS = ["fr_hs1", "fr_hs2", "fr_hsr", "fr_ds"]
tap_id = "fr_hs1"   # change manually: the set whose figures are shown

fr_fits, fr_band_data = {}, {}
for t in FR_TAPS:
    fr_fits[t], bds = full_band_mpe(
        [(frf[t].freqs, frf[t].H)], resp_win[t].alpha, fs, BANDS, spectrum="frf_tap",
        quantity="acceleration", ordmax=ORDMAX, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="global", cache=t, cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    fr_band_data[t] = bds[0]   # [BandFit, ...] of the one pass
    print(f"{t}: {fr_fits[t].model.Fn.size:2d} modes @ {jnp.round(fr_fits[t].model.Fn, 1)}")

# the modes each band of tap_id contributed
for flo, fhi, est, poles, res in fr_band_data[tap_id]:
    fn = res.Fn[(res.Fn >= flo) & (res.Fn <= fhi)]
    print(f"{tap_id} band {flo:4g}-{fhi:4g} Hz (fs_fake={est.fs:5g}): {len(fn)} modes @ {jnp.round(fn, 1)}")

Each sub-band is fitted on its own normalised axis, so it gets its own stabilisation chart. The right axis is model order; the CMIF of that band's FRF is overlaid on the left. 

In [ ]:
# One stabilisation chart per sub-band, the CMIF of the band's FRF overlaid.
# est.freqs already holds the band's physical frequencies, so we slice the FRF to match
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_plot(poles, data=frf[t].H[:, :, sel], indicator="cmif", xlim=(flo, fhi),
                                  figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/stab_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

Within each band the stable poles are grouped into modes with DBSCAN on the modal distance (normalised frequency gap + `1 − MAC`), and each cluster's representative pole is kept. The kept poles from every band are then combined into one pole set. 

In [ ]:
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_cluster_plot(res, data=frf[t].H[:, :, sel], xlim=(flo, fhi), cmap_name="Dark2",
                                          figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/cluster_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

To pick `eps` (how close two poles must be, in modal distance, to count as the same mode), we use the k-distance graph: for every pole, measure its modal distance to its `min_pts`-th nearest neighbour, then sort these distances from largest to smallest.

Poles that belong to a real mode sit close together across model orders, so their distances are small — this shows up as a flat, low plateau on the right. Poles that are just noise are scattered, so their distances are larger — a steep rise on the left. Where the two meet (the "knee" of the curve) is a good place to set `eps`.

In [ ]:
# k-distance curve per band: the knee is a good eps
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        fig, axes = est.k_distance_plot(poles, min_pts=MIN_PTS, eps=EPS, figsize=(6, 4),
                                        save_fig_name=FIG_DIR / f"ema/{t}/kdist_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# Frequency-damping cluster chart per band;
# a tight vertical group means consistent damping across model orders.
for t in FR_TAPS:
    for flo, fhi, est, poles, res in fr_band_data[t]:
        fig, axes = est.cluster_freqs_zetas_plot(res, xlim=(flo, fhi), figsize=(6, 4),
                                                 save_fig_name=FIG_DIR / f"ema/{t}/fzeta_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

#### LSFD

The combined poles fix the frequencies and damping. LSFD holds them fixed and fits the modal constants (residues $A$) plus the out-of-band lower/upper residuals, in one encompassing band over the whole identified range (2–560 Hz) (see Avitabile Fig. 9.18). Fitting the residues jointly over the full band (rather than per-band) gives the tightest overall synthesis. `full_band_mpe` ran this fit above: each `fr_fits[t]` holds the LSFD estimator and the modal model it fitted.

In [ ]:
# the LSFD fit of tap_id; it holds the combined Fns and Zetas
lsfd, model = fr_fits[tap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

In [ ]:
# Measured vs synthesised FRF, one panel per sensor
for t in FR_TAPS:
    lsfd, model = fr_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        frf[t].H,
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|H|$ [(m/s$^2$)/N]" for lab in SENSORS],
        # phase=True,
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"ema/{t}/synthesis.pdf",
    )
    if t == tap_id:
        plt.show()
    else:
        plt.close(fig)

# Unit-normalised real shapes of tap_id
model = fr_fits[tap_id].model
shapes = model.mode_shapes(real=True, normalize="l2")  # (Q, M)
print("unit-normalised real mode shapes (Q x M):", shapes.shape)

# values: one row per sensor, one column per identified mode (labelled by fn)
print(f"{'sensor / f [Hz]':>16}" + "".join(f"{f:>9.1f}" for f in model.Fn))
for s, row in zip(SENSORS, shapes):
    print(f"{s:>16}" + "".join(f"{v:>9.4f}" for v in row))

The modes of every front set in one table, written to `outputs/garteur/` in CSV, Excel and HTML via `reporting.mode_table` and `export.save_table`.

In [ ]:
# one row per identified mode
fr_tap_df = rp.mode_table(fr_fits)
display(fr_tap_df.head())

# archive under OUT_DIR in CSV, Excel and HTML
save_table(fr_tap_df, OUT_DIR / "fr_tap_modes", fmt=["csv", "excel", "html"])

### MPE for R-wing taps

The same procedure as for the front sets, with one band more and a higher maximum order. The figures are saved under `figures/garteur/ema/<set>/` as well.

In [ ]:
RW_TAPS = ["rw_hs2", "rw_hsr", "rw_ds"]
tap_id = "rw_ds"   # change manually: the set whose figures are shown

# the last front band split in two, and a higher order
BANDS_RW_TAP = [
    (2, 120, 300.0),
    (120, 320, 800.0),
    (320, 470, 1200.0),
    (470, 560, 1400.0),
]
ORDMAX_RW_TAP = 150

rw_fits, rw_band_data = {}, {}
for t in RW_TAPS:
    rw_fits[t], bds = full_band_mpe(
        [(frf[t].freqs, frf[t].H)], resp_win[t].alpha, fs, BANDS_RW_TAP, spectrum="frf_tap",
        quantity="acceleration", ordmax=ORDMAX_RW_TAP, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="global", cache=t, cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    rw_band_data[t] = bds[0]   # [BandFit, ...] of the one pass
    print(f"{t}: {rw_fits[t].model.Fn.size:2d} modes @ {jnp.round(rw_fits[t].model.Fn, 1)}")

# the modes each band of tap_id contributed
for flo, fhi, est, poles, res in rw_band_data[tap_id]:
    fn = res.Fn[(res.Fn >= flo) & (res.Fn <= fhi)]
    print(f"{tap_id} band {flo:4g}-{fhi:4g} Hz (fs_fake={est.fs:5g}): {len(fn)} modes @ {jnp.round(fn, 1)}")

In [ ]:
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_plot(poles, data=frf[t].H[:, :, sel], indicator="cmif", xlim=(flo, fhi),
                                  figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/stab_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        sel = (frf[t].freqs >= flo) & (frf[t].freqs <= fhi)
        fig, axes = est.stab_cluster_plot(res, data=frf[t].H[:, :, sel], xlim=(flo, fhi), cmap_name="Dark2",
                                          figsize=(6, 4), save_fig_name=FIG_DIR / f"ema/{t}/cluster_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# k-distance curve per band: the knee is a good eps
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        fig, axes = est.k_distance_plot(poles, min_pts=MIN_PTS, eps=EPS, figsize=(6, 4),
                                        save_fig_name=FIG_DIR / f"ema/{t}/kdist_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# Frequency-damping cluster chart per band;
# a tight vertical group means consistent damping across model orders.
for t in RW_TAPS:
    for flo, fhi, est, poles, res in rw_band_data[t]:
        fig, axes = est.cluster_freqs_zetas_plot(res, xlim=(flo, fhi), figsize=(6, 4),
                                                 save_fig_name=FIG_DIR / f"ema/{t}/fzeta_{flo:g}-{fhi:g}.pdf")
        if t == tap_id:
            plt.show()
        else:
            plt.close(fig)

In [ ]:
# the LSFD fit of tap_id; it holds the combined Fns and Zetas
lsfd, model = rw_fits[tap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

In [ ]:
for t in RW_TAPS:
    lsfd, model = rw_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        frf[t].H,
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|H|$ [(m/s$^2$)/N]" for lab in SENSORS],
        # phase=True,
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"ema/{t}/synthesis.pdf",
    )
    if t == tap_id:
        plt.show()
    else:
        plt.close(fig)

# Unit-normalised real shapes of tap_id
model = rw_fits[tap_id].model
shapes = model.mode_shapes(real=True, normalize="l2")  # (Q, M)
print("unit-normalised real mode shapes (Q x M):", shapes.shape)

# values: one row per sensor, one column per identified mode (labelled by fn)
print(f"{'sensor / f [Hz]':>16}" + "".join(f"{f:>9.1f}" for f in model.Fn))
for s, row in zip(SENSORS, shapes):
    print(f"{s:>16}" + "".join(f"{v:>9.4f}" for v in row))

In [ ]:
# one row per identified mode
rw_tap_df = rp.mode_table(rw_fits)
display(rw_tap_df.head())

# archive under OUT_DIR in CSV, Excel and HTML
save_table(rw_tap_df, OUT_DIR / "rw_tap_modes", fmt=["csv", "excel", "html"])

# Dummy OMA

In [ ]:
ftaps = {
    "nb_hs": data["fanfr_nb_hs"],
    "nb_ds": data["fanfr_nb_ds"],
    "b0_hs": data["fanfr_b0_hs"],
    "b0_ds": data["fanfr_b0_ds"],
    "b45_hs": data["fanfr_b45_hs"],
    "b45_ds": data["fanfr_b45_ds"],
}

Let's plot the response signals

In [ ]:
cmap = plt.get_cmap("Dark2")

figs = {}
# for name, tap in ftaps.items():
for name, ftap in {k: ftaps[k] for k in ["nb_hs"]}.items():
    n_reps = ftap.data.shape[0]
    n_cols = len(ftap.channels[1:])

    fig, axes = plt.subplots(
        n_reps, n_cols,
        figsize=(4 * n_cols, 2 * n_reps),
        sharex=True, squeeze=False,
    )
    for j, ch in enumerate(ftap.channels[1:]):
        sig = ftap.load_channel(ch)
        color = cmap(j)   # one color per sensor 
        for i in range(n_reps):
            axes[i, j].plot(time_array, sig[i, :], color=color, linewidth=1.0)
            axes[i, j].grid(True, linestyle="--", alpha=0.7)
        axes[0, j].set_title(ch)

    for i in range(n_reps):
        axes[i, 0].set_ylabel(f"rep {i + 1}")
    for j in range(n_cols):
        axes[-1, j].set_xlabel("Time [s]")

    fig.suptitle(name)
    fig.tight_layout()
    save_path = FIG_DIR / f"oma/responses_{name}.pdf"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=300, bbox_inches="tight")
    figs[name] = (fig, axes)

### Preprocess the data at hand

The raw records are sampled well above the modal band. Two steps clean up the band:

1. *detrend + decimate* to lower the sample rate while keeping the modal band below the new Nyquist.
2. *zero-phase highpass* to strip any low-frequency drift. Zero-phase filtering keeps the relative phase between channels, which mode shapes depend on.

In [ ]:
from modekit.dataprep import preprocess

Q_DEC = 8  # 16384 Hz -> 2048 Hz; Nyquist 1024 Hz keeps the 2-560 Hz band (used in EMA)
ftaps_proc = {}
for name, ftap in ftaps.items():
    y = preprocess.detrend_data(ftap.data[:, 1:, :], type="constant")  # responses only
    y, fs_dec = preprocess.decimate_data(y, fs, Q_DEC)
    y = preprocess.filter_data(y, fs_dec, Wn=5.0, order=8, btype="highpass")
    ftaps_proc[name] = y

print(f"decimated: fs = {fs_dec:g} Hz, shape = {ftaps_proc['nb_hs'].shape}")

In [ ]:
from modekit.freq_models import OmaModel
import equinox as eqx

# n_lags << nt; half-spectra resolution fs_dec / (2 * n_lags) = 0.25 Hz
ftap_models = {
    name: OmaModel(ftaps_proc[name], fs_dec, fft_args={"n_lags": 4096}, estimator="cor")
    for name in ftaps_proc
}

print("\nOmaModel fields:")
print("-" * 30)
eqx.tree_pprint(ftap_models["nb_hs"])

### Autocorrelation of the responses

By the Wiener–Khinchin theorem the autocorrelation and the power spectrum are a Fourier pair, so under broadband excitation the output correlation functions read like free decays — the same information as an impulse response. That is the value of the correlogram: it means we can use classical time-domain modal analysis algorithms on output-only data by converting it to correlation functions first.

`get_ac` returns the same correlations the `'cor'` half-spectra are built from (before the exponential window). The general rule for `n_lags`: long enough that the correlations decay enough, but only a small fraction of the record length (i.e., 10–20% of `nt`) because the biased estimate averages fewer products as the lag grows, so its variance blows up as the lag approaches `nt`.

In [ ]:
AC = namedtuple("AC", ["lags", "R"])
acs = {name: AC(*m.get_ac(n_lags=4096, normalize=True)) for name, m in ftap_models.items()}

fig, axes = plot_signal(
    np.asarray(acs["nb_hs"].lags),
    {name: np.asarray(ac.R) for name, ac in acs.items()},
    x_label=r"Lag $\tau$ [s]",
    y_labels=[rf"$\rho_{{{q}}}(\tau)$" for q in range(1, np.shape(acs["nb_hs"].R)[0] + 1)],
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/autocorrelation.pdf",
)

### Output spectra after preprocessing the data

The output PSDs through `get_psd` (diagonal of $S_{yy} = S^+ + S^{+H}$), as for the `EmaModel`.

In [ ]:
OPSD = namedtuple("OPSD", "freqs Syy")
opsd = {name: OPSD(*m.get_psd()) for name, m in ftap_models.items()}

# abs: the correlogram diagonal can dip below zero in the decimation
fig, axes = plot_signal(
    opsd["nb_hs"].freqs,
    {name: np.abs(p.Syy) for name, p in opsd.items()},
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{y_{q}y_{q}}}(f)$ [(m/s$^2$)$^2$]" for q in range(1, opsd["nb_hs"].Syy.shape[0] + 1)],
    yscale="log",
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy.pdf",
)
plt.show()

# zoomed-view on the identified band
fig, axes = plot_signal(
    opsd["nb_hs"].freqs,
    {name: np.abs(p.Syy) for name, p in opsd.items()},
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{y_{q}y_{q}}}(f)$ [(m/s$^2$)$^2$]" for q in range(1, opsd["nb_hs"].Syy.shape[0] + 1)],
    yscale="log",
    xlim=(5, 400),
    ylim=(1e-5, 10),
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy_zoom.pdf",
)
plt.show()

### Periodogram vs correlogram spectra

The same processed records through both estimators. `'per'` is Welch averaging (hann, 4096-sample segments, 50% overlap: df = 0.5 Hz with ~16 averages), `'cor'` the exponential-windowed correlogram (df = 0.25 Hz, drawn on the shared 0.5 Hz grid and scaled by $2/f_s$ to density units). The correlogram's window damping flattens and widens the peak (energy conserved, so the line's height drops and the surrounding baseline rises). pLSCF's `sd_cor` correction removes that trade, while Welch keeps the sharper peak at the cost of a noisier baseline.

In [ ]:
psd_per = OPSD(*OmaModel(
    ftaps_proc["nb_hs"], fs_dec, fft_args={"nperseg": 4096}, estimator="per"
).get_psd())
psd_cor = opsd["nb_hs"]

fig, axes = plot_signal(
    psd_per.freqs,
    {
        "per (Welch)": np.asarray(psd_per.Syy),
        # correlogram scaled to density units [(m/s^2)^2/Hz]
        "cor (correlogram)": np.abs(psd_cor.Syy[:, ::2]) * 2 / fs_dec,
    },
    x_label="Frequency [Hz]",
    y_labels=[rf"$S_{{y_{q}y_{q}}}(f)$ [(m/s$^2$)$^2$/Hz]" for q in range(1, np.shape(psd_per.Syy)[0] + 1)],
    yscale="log",
    xlim=(5, 400),
    ylim=(1e-8, 1e-2),
    figsize=(6, 8),
    save_fig_name=FIG_DIR / "oma/Syy_per_vs_cor_nb_hs.pdf",
)
plt.show()

### pLSCF on the ambient half-spectra

Same band-wise procedure as the EMA sets, with the half-spectra in the role of the FRFs and `spectrum="sd_cor"` so pLSCF removes exactly the artificial damping of the 1% exponential window (see `get_output_spectra`). The references are fitted in two separate passes, the front sensor (`[0]`) and the wing pair (`[2, 3]`). The two mode sets are merged by nearest frequency; a mode found by both passes is counted once and kept from whichever pass identifies it more robustly, i.e., the pole with the larger stabilisation-cluster confidence. The tail stays an output row throughout (its mode-shape component is kept). The residue (LSFD) fit then uses the merged poles on the spectra against the union of the reference columns. No band fit carries a pole's factors on that union, because the passes used different references. So `full_band_mpe` re-estimates them with a local pLSCF model per merged pole on the union spectra (`refs="local"`), and the LSFD solves for the shapes only. Every fan set's figures are saved under `figures/garteur/oma/<set>/`; the set named by `ftap_id` is shown.

In [ ]:
OS = namedtuple("OS", "freqs S")

# two reference passes, fitted separately then merged; tail = output row only
REF_PASSES = {
    "front": [0],
    "wings": [2, 3],
}
REFS_LSFD = [c for refs in REF_PASSES.values() for c in refs]   # union -> LSFD residue fit

# per fan set: the half-spectra of each pass, and of the union for the LSFD
os_pass = {
    name: {p: OS(*m.get_output_spectra(ref_dofs=refs)) for p, refs in REF_PASSES.items()}
    for name, m in ftap_models.items()
}
os_lsfd = {name: OS(*m.get_output_spectra(ref_dofs=REFS_LSFD)) for name, m in ftap_models.items()}

W_RATE = ftap_models["nb_hs"].window_rate  # window damping [1/s]
BANDS_FTAP = [
    (5, 120, 300.0),
    (120, 320, 800.0),
    (320, 400, 1000.0),
]
ORDMAX_FTAP = 150
# the local pLSCF models that give the factors; at order 2 they miss weak modes
# that only the wing references find, so order 4
LOCAL_KW = dict(order=4, deltaf=2.0, rtol=0.03)

In [ ]:
FTAPS = ["nb_hs", "nb_ds"]
ftap_id = "nb_ds"   # change manually: the set whose figures are shown

# the procedure on every fitted fan set: one band-wise pLSCF per reference pass,
# the passes' poles merged, then the LSFD on the union spectra
ftap_fits, ftap_band_data = {}, {}
for t in FTAPS:
    ftap_fits[t], bds = full_band_mpe(
        [(os_pass[t][p].freqs, os_pass[t][p].S) for p in REF_PASSES], W_RATE, fs_dec, BANDS_FTAP,
        spectrum="sd_cor", lsfd_data=(os_lsfd[t].freqs, os_lsfd[t].S),
        ordmax=ORDMAX_FTAP, sc=SC, hc=HC, eps=EPS, min_pts=MIN_PTS,
        refs="local", local_kw=LOCAL_KW,
        cache=[f"{t}_{p}" for p in REF_PASSES], cache_dir=MPE_CACHE_DIR, refit=REFIT_MPE)
    ftap_band_data[t] = list(zip(REF_PASSES, bds))   # [(pass, [BandFit, ...]), ...]
    fn = np.asarray(ftap_fits[t].model.Fn)
    print(f"{t}: {int(np.isfinite(fn).sum()):2d} modes @ {np.round(fn, 1)}")

# the modes each pass of ftap_id found, before the merge
for p, bd in ftap_band_data[ftap_id]:
    fn = np.concatenate([np.asarray(res.Fn)[np.asarray((res.Fn >= flo) & (res.Fn <= fhi))]
                         for flo, fhi, est, poles, res in bd])
    print(f"{ftap_id} {p:>5} refs: {fn.size:2d} modes @ {np.round(fn, 1)}")

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        sp = os_pass[t][p]
        for flo, fhi, est, poles, res in bd:
            sel = (sp.freqs >= flo) & (sp.freqs <= fhi)
            fig, axes = est.stab_plot(
                poles, data=np.asarray(sp.S)[:, :, sel], indicator="cmif",
                xlim=(flo, fhi), figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/stab_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        sp = os_pass[t][p]
        for flo, fhi, est, poles, res in bd:
            sel = (sp.freqs >= flo) & (sp.freqs <= fhi)
            fig, axes = est.stab_cluster_plot(
                res, data=np.asarray(sp.S)[:, :, sel], xlim=(flo, fhi),
                cmap_name="Dark2", figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/cluster_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
for t in FTAPS:
    for p, bd in ftap_band_data[t]:
        for flo, fhi, est, poles, res in bd:
            fig, axes = est.cluster_freqs_zetas_plot(
                res, xlim=(flo, fhi), figsize=(6, 4),
                save_fig_name=FIG_DIR / f"oma/{t}/fzeta_{p}_{flo:g}-{fhi:g}.pdf")
            fig.suptitle(f"{p} refs", y=1.02)
            if t == ftap_id:
                plt.show()
            else:
                plt.close(fig)

In [ ]:
# the LSFD fit of ftap_id: the merged poles over the full band, on the spectra
# against the union of the reference columns
lsfd, model = ftap_fits[ftap_id]
print("fn   [Hz]:", jnp.round(model.Fn, 2))
print("zeta [%] :", jnp.round(model.Zeta * 100, 3))

for t in FTAPS:
    lsfd, model = ftap_fits[t]
    fig, axes = lsfd.synthesis_plot(
        model,
        os_lsfd[t].S,
        pairs=[(q, 0) for q in range(len(SENSORS))],  # every output against the front reference
        xlim=lsfd.band,
        y_labels=[f"{lab}\n" + r"$|S^+|$" for lab in SENSORS],
        figsize=(6, 8),
        save_fig_name=FIG_DIR / f"oma/{t}/synthesis.pdf",
    )
    if t == ftap_id:
        plt.show()
    else:
        plt.close(fig)

In [ ]:
# one row per identified mode, across the fitted fan sets
ftap_df = rp.mode_table(ftap_fits)
display(ftap_df.head(12))

save_table(ftap_df, OUT_DIR / "fan_tap_modes", fmt=["csv", "excel", "html"])